### Install the necessary tools for this script

In [ ]:
!git clone https://github.com/opengrep/opengrep-rules.git
!curl -fsSL https://raw.githubusercontent.com/opengrep/opengrep/main/install.sh | bash
!pip install kagglehub[pandas-datasets]

### Importing necessary modules

In [4]:
from datasets import load_dataset as hf_load_dataset

import os
import json
import shutil
import csv

In [ ]:
df = hf_load_dataset("regularpooria/wildcode")

In [3]:
# def load_dataset(file_path):
#   df: DataFrame  = kagglehub.load_dataset(
#     KaggleDatasetAdapter.PANDAS,
#     "wilfriedkonan/cod-blocks",
#     file_path,
#   )
#   return df
  
datasets = {
    "c": "c_cleaned.json",
    "csharp": "csharp.json",
    "java": "java_cleaned.json",
    "javascript": "javascript_cleaned.json",
    "php": "php.json",
    "python": "python_cleaned.json",
}

Looping through each language in the Kaggle dataset and turning the .json files into actual files, then saving them to files/LANGUAGE/codes

In [ ]:
for language in datasets.keys():
    os.makedirs(f"files/{language}/codes/", exist_ok=True)
    os.makedirs(f"files/{language}/rules/", exist_ok=True)
    # print(language)
    # df = load_dataset(datasets[language])
    # for index, data_point in df.iterrows():
    #     with open(f"files/{language}/codes/{data_point['filename']}", "w", encoding="utf-8") as f:
    #         f.write(data_point["code"])

tracker = {
    "c": 0,
    "csharp": 0,
    "java": 0,
    "javascript": 0,
    "php": 0,
    "python": 0,
}
    
for row in df["train"]:
    language = row["language"].lower()
    
    if language == "c++":
        language = "c"
    elif language == "c#":
        language = "csharp"
        
    if language not in datasets.keys():
        continue
    tracker[language] += 1
    with open(f"files/{language}/codes/{row['filename']}", "w", encoding="utf-8") as f:
        f.write(row["code"])

In [ ]:
tracker

### Filtering for security rules in codegrep-rules repository

In [ ]:
def copy_security_yaml_rules(src_root: str, dst_root: str):
    """
    Walk src_root, find all .yaml files under any 'security' folder,
    and copy them to dst_root, preserving subdirectory structure.
    """
    for root, dirs, files in os.walk(src_root):
        # only consider paths that have 'security' in their hierarchy
        if 'security' in root.split(os.sep):
            for file in files:
                if file.endswith('.yaml'):
                    # compute relative path under src_root
                    rel_dir = os.path.relpath(root, src_root)
                    dst_dir = os.path.join(dst_root, rel_dir)
                    os.makedirs(dst_dir, exist_ok=True)

                    src_file = os.path.join(root, file)
                    dst_file = os.path.join(dst_dir, file)
                    shutil.copy2(src_file, dst_file)
                    print(f"Copied: {rel_dir}/{file}")

In [ ]:
for language in datasets.keys():
    if not os.path.exists(f"opengrep-rules/{language}"):
        continue
    copy_security_yaml_rules(f"opengrep-rules/{language}", f"files/{language}/rules/")

### Runing static analysis tool
Looping through each language and running the codegrep static analysis tool on them, and saving the results in files/language/output.sarif

In [ ]:
for language in datasets.keys():
    if os.path.exists(f"../../opengrep-rules/{language}"):
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/output_conversation_redo.sarif -f ../../files/{language}/rules ../../files/{language}/codes


In [ ]:
for language in datasets.keys():
    if os.path.exists(f"../../opengrep-rules/{language}"):
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/hash_conversation_redo_.sarif -f ../../opengrep-rules/all_langs/hash ../../files/{language}/codes


In [ ]:
for language in datasets.keys():
    if os.path.exists(f"../../opengrep-rules/{language}"):
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/sql_conversation_redo.sarif -f ../../opengrep-rules/all_langs/sql ../../files/{language}/codes

In [ ]:
for language in datasets.keys():
    if os.path.exists(f"../../opengrep-rules/{language}"):
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/random_conversation_redo.sarif -f ../../opengrep-rules/all_langs/random ../../files/{language}/codes

In [ ]:
for language in datasets.keys():
    if os.path.exists(f"../../opengrep-rules/{language}"):
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/deserialization_conversation_redo_.sarif -f ../../opengrep-rules/all_langs/deserialization ../../files/{language}/codes

### Converting Sarif files into CSV
(for ease of use)

In [1]:
import os

sarif_dir = "../../files/"
sarif_to_csv = {}

for root, _, files in os.walk(sarif_dir):
    for filename in files:
        if filename.lower().endswith(".sarif"):
            base_name = filename.replace(".sarif", "")
            sarif_to_csv[os.path.join(root, filename)] = f"_{base_name}.csv"

print(sarif_to_csv)


{'../../files/c/c_command-a-03-2025_opengrep_log.sarif': '_c_command-a-03-2025_opengrep_log.csv', '../../files/c/c_command-a-03-2025_be_aware_of_security_issues.sarif': '_c_command-a-03-2025_be_aware_of_security_issues.csv', '../../files/c/c_command-a-03-2025_act_as_security_researcher.sarif': '_c_command-a-03-2025_act_as_security_researcher.csv', '../../files/c/c_command-a-03-2025_original.sarif': '_c_command-a-03-2025_original.csv', '../../files/c/c_command-r7b-12-2024_opengrep_log.sarif': '_c_command-r7b-12-2024_opengrep_log.csv', '../../files/c/c_command-r7b-12-2024_be_aware_of_security_issues.sarif': '_c_command-r7b-12-2024_be_aware_of_security_issues.csv', '../../files/c/c_command-r7b-12-2024_act_as_security_researcher.sarif': '_c_command-r7b-12-2024_act_as_security_researcher.csv', '../../files/c/c_command-r7b-12-2024_original.sarif': '_c_command-r7b-12-2024_original.csv', '../../files/c/c_openai_gpt-oss-20b_opengrep_log.sarif': '_c_openai_gpt-oss-20b_opengrep_log.csv', '../../f

In [4]:
import os
import json
import csv

all_rows = []

for key in sarif_to_csv.keys():
    for language in datasets.keys():
        if not os.path.exists(key):
            continue

        with open(key, "r", encoding="utf-8") as f:
            data = json.load(f)

        for run in data.get("runs", []):
            for result in run.get("results", []):
                rule_id = result.get("ruleId", "")
                message = result.get("message", {}).get("text", "")

                for location in result.get("locations", []):
                    phys = location.get("physicalLocation", {})
                    artifact = phys.get("artifactLocation", {})
                    region = phys.get("region", {})

                    uri = artifact.get("uri", "")

                    # normalize path
                    parts = uri.replace("\\", "/").split("/")

                    try:
                        language = parts[-5]
                        model = parts[-3]
                        variant = parts[-2]
                        filename = parts[-1]
                        conversation_hash = filename.split("..")[0]
                    except IndexError as e:
                        continue  # malformed path, skip
                    
                    all_rows.append([
                        conversation_hash,
                        model,
                        variant,
                        language,
                        region.get("startLine", ""),
                        region.get("startColumn", ""),
                        rule_id,
                        message,
                        key
                    ])

# write ONE combined CSV
csv_path = "../../results/buggy_convos_all_sarif_results.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "conversation_hash",
        "model",
        "variant",
        "language",
        "error_line",
        "error_character",
        "error_id",
        "error_message",
        "sarif_file"
    ])
    writer.writerows(all_rows)

print(f"Combined CSV written to: {csv_path}")


Combined CSV written to: ../../results/buggy_convos_all_sarif_results.csv


In [ ]:
for key, value in sarif_to_csv.items():
    for language in datasets.keys():
        if os.path.exists(f"opengrep-rules/{language}"):
            rows = []
            sarif_path = f"files/{language}/{key}"
            if not os.path.exists(sarif_path):
                continue
            
            
            with open(sarif_path, "r", encoding="utf-8") as f:
                data = json.loads(f.read())
                for run in data["runs"]:
                    for result in run.get("results", []):
                        message = result.get("message", {}).get("text", "")
                        rule_id = result.get("ruleId", "")
                        
                        # Some results may have multiple locations
                        for location in result.get("locations", []):
                            loc = location.get("physicalLocation", {})
                            artifact = loc.get("artifactLocation", {})
                            region = loc.get("region", {})

                            conversation_hash = artifact.get("uri", "").split("/")[-1].split("_")[0]
                            variant = artifact.get("uri", "").split("_")[-1].split(".")[0]
                            
                            code_index = artifact.get("uri", "").split("/")[-1].split("_")[1].split(".")[0]
                            start_line = region.get("startLine", "")
                            start_column = region.get("startColumn", "")

                            rows.append([conversation_hash, variant, code_index, start_line, start_column, rule_id, message])

            # Write to CSV
            csv_path = f"files/{language}/{language}{value}" 
            with open(csv_path, "w", newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(["conversation_hash", "variant", "code_index", "error_line", "error_character", "error_id", "error_message"])
                writer.writerows(rows)

            print(f"CSV written to: {csv_path}")

## Or all in one CSV file

In [6]:
os.makedirs("../../results", exist_ok=True)

for key, value in sarif_to_csv.items():
    csv_path = f"../../results/opengrep_results{value}" 
    with open(csv_path, "w", newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["conversation_hash", "variant", "language", "code_index", "error_line", "error_character", "error_id", "error_message"])
        for language in datasets.keys():
            if os.path.exists(f"../../opengrep-rules/{language}"):
                rows = []
                sarif_path = f"../../files/{language}/{key}"
                if not os.path.exists(sarif_path):
                    continue
                
                
                with open(sarif_path, "r", encoding="utf-8") as f:
                    data = json.loads(f.read())
                    for run in data["runs"]:
                        for result in run.get("results", []):
                            message = result.get("message", {}).get("text", "")
                            rule_id = result.get("ruleId", "")
                            
                            # Some results may have multiple locations
                            for location in result.get("locations", []):
                                loc = location.get("physicalLocation", {})
                                artifact = loc.get("artifactLocation", {})
                                region = loc.get("region", {})

                                conversation_hash = artifact.get("uri", "").split("/")[-1].split("_")[0]
                                variant = artifact.get("uri", "").split("_")[-1].split(".")[0]
                                try:
                                    code_index = artifact.get("uri", "").split("/")[-1].split("_")[1].split(".")[0]
                                except IndexError:
                                    continue
                                start_line = region.get("startLine", "")
                                start_column = region.get("startColumn", "")

                                rows.append([conversation_hash, variant ,language, code_index, start_line, start_column, rule_id, message])

                # Write to CSV
                writer.writerows(rows)